# CodeMix-T — Phase 5: Evaluation

Covers:
1. Load best trained model
2. BLEU score (sacrebleu)
3. chrF++ score
4. Baseline comparisons (mBART-50, Google Translate)
5. Human evaluation template
6. Error analysis
7. Results table for paper

## Cell 1 — Install & Setup

In [ ]:
!pip install -q sacrebleu sentencepiece transformers torch

import torch, json, os
import pandas as pd
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF
from tqdm import tqdm
import sentencepiece as spm
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/CodeMixT'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Cell 2 — Load Best Model & Tokenizer

In [ ]:
# Paste architecture code from Phase 2 here (CodeMixTConfig through CodeMixT)
# Then load:

with open(f'{DRIVE_DIR}/model/config.json') as f:
    cfg = CodeMixTConfig(**json.load(f))

model = CodeMixT(cfg).to(device)
ckpt  = torch.load(f'{DRIVE_DIR}/checkpoints/codemix_t_best.pt', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Best model loaded | val_loss={ckpt["val_loss"]:.4f}')

# Load tokenizer
sp = spm.SentencePieceProcessor()
sp.load(f'{DRIVE_DIR}/tokenizer/codemix_bpe.model')
print(f'Tokenizer loaded: vocab_size={sp.get_piece_size()}')

# Instantiate chatbot (includes beam search)
chatbot = CodeMixTChatbot(model, f'{DRIVE_DIR}/tokenizer/codemix_bpe.model', cfg)

## Cell 3 — Load Test Set

In [ ]:
# Load test set (created in Phase 1)
df_test = pd.read_json(f'{DRIVE_DIR}/data_final/test.jsonl', lines=True)
print(f'Test samples: {len(df_test)}')
print(df_test[['source', 'target', 'language']].head(5).to_string())

# Split by language for per-language evaluation
df_hinglish = df_test[df_test['language'].isin(['hinglish', 'hindi'])]
df_tanglish = df_test[df_test['language'] == 'tanglish']
print(f'\nHinglish test: {len(df_hinglish)}')
print(f'Tanglish test: {len(df_tanglish)}')

## Cell 4 — Generate Translations

In [ ]:
def generate_translations(df, chatbot, desc='Translating'):
    """Generate translations for a DataFrame subset."""
    predictions = []
    references  = []
    sources     = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        pred = chatbot.translate(row['source'])
        predictions.append(pred)
        references.append(row['target'])
        sources.append(row['source'])

    return sources, predictions, references


print('Generating translations on full test set...')
src_all, pred_all, ref_all = generate_translations(df_test, chatbot, 'Full test set')

print('Generating translations on Hinglish subset...')
src_hi, pred_hi, ref_hi = generate_translations(df_hinglish, chatbot, 'Hinglish')

print('Generating translations on Tanglish subset...')
src_ta, pred_ta, ref_ta = generate_translations(df_tanglish, chatbot, 'Tanglish')

print('Done.')

## Cell 5 — BLEU & chrF++ Scores (Our Model)

In [ ]:
bleu  = BLEU()
chrf  = CHRF(word_order=2)  # word_order=2 → chrF++

def score(predictions, references, label=''):
    """Compute BLEU and chrF++ scores."""
    # sacrebleu expects: corpus_score(hypotheses, [references_list])
    b = bleu.corpus_score(predictions, [references])
    c = chrf.corpus_score(predictions, [references])
    print(f'{label}')
    print(f'  BLEU:   {b.score:.2f}')
    print(f'  chrF++: {c.score:.2f}')
    return b.score, c.score


print('=== CodeMix-T Scores ===')
bleu_all,  chrf_all  = score(pred_all, ref_all,  'Full test set')
bleu_hi,   chrf_hi   = score(pred_hi,  ref_hi,   'Hinglish only')
bleu_ta,   chrf_ta   = score(pred_ta,  ref_ta,   'Tanglish only')

codemix_scores = {
    'Model': 'CodeMix-T (ours)',
    'BLEU (all)':      round(bleu_all, 2),
    'chrF++ (all)':    round(chrf_all, 2),
    'BLEU (Hinglish)': round(bleu_hi, 2),
    'BLEU (Tanglish)': round(bleu_ta, 2),
}

## Cell 6 — Baseline: mBART-50 (zero-shot)

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

print('Loading mBART-50 (this may take a few minutes)...')
mbart_model = MBartForConditionalGeneration.from_pretrained('facebook/mbart-large-50-many-to-many-mmt')
mbart_tok   = MBart50TokenizerFast.from_pretrained('facebook/mbart-large-50-many-to-many-mmt')
mbart_model.to(device)
mbart_model.eval()
print('mBART-50 loaded.')


def mbart_translate(texts, src_lang='hi_IN', tgt_lang='en_XX', batch_size=16):
    """
    Translate a list of texts using mBART-50.
    Uses Hindi (hi_IN) as source — closest to Hinglish.
    For Tanglish, use ta_IN.
    """
    mbart_tok.src_lang = src_lang
    translations = []

    for i in tqdm(range(0, len(texts), batch_size), desc='mBART'):
        batch = texts[i:i+batch_size]
        encoded = mbart_tok(batch, return_tensors='pt', padding=True,
                            truncation=True, max_length=128).to(device)
        with torch.no_grad():
            generated = mbart_model.generate(
                **encoded,
                forced_bos_token_id=mbart_tok.lang_code_to_id[tgt_lang],
                num_beams=4, max_length=128
            )
        decoded = mbart_tok.batch_decode(generated, skip_special_tokens=True)
        translations.extend(decoded)

    return translations


print('Running mBART-50 on full test set...')
mbart_preds_all = mbart_translate(src_all, src_lang='hi_IN')

print('Running mBART-50 on Hinglish subset...')
mbart_preds_hi = mbart_translate(src_hi, src_lang='hi_IN')

print('Running mBART-50 on Tanglish subset (using ta_IN)...')
mbart_preds_ta = mbart_translate(src_ta, src_lang='ta_IN')

print('\n=== mBART-50 Scores ===')
mbart_bleu_all, mbart_chrf_all = score(mbart_preds_all, ref_all,  'Full test set')
mbart_bleu_hi,  mbart_chrf_hi  = score(mbart_preds_hi,  ref_hi,   'Hinglish only')
mbart_bleu_ta,  mbart_chrf_ta  = score(mbart_preds_ta,  ref_ta,   'Tanglish only')

mbart_scores = {
    'Model': 'mBART-50 (zero-shot)',
    'BLEU (all)':      round(mbart_bleu_all, 2),
    'chrF++ (all)':    round(mbart_chrf_all, 2),
    'BLEU (Hinglish)': round(mbart_bleu_hi, 2),
    'BLEU (Tanglish)': round(mbart_bleu_ta, 2),
}

del mbart_model
torch.cuda.empty_cache()

## Cell 7 — Baseline: No-LangID Ablation Scores

In [ ]:
# Load the no-LangID ablation model (trained in Phase 4)
# This is the clearest evidence for your paper that LangID embeddings help

no_lid_cfg = CodeMixTConfig(**{**json.load(open(f'{DRIVE_DIR}/model/config.json')), 'lang_embed_dim': 0})
no_lid_model = CodeMixT(no_lid_cfg).to(device)
no_lid_model.encoder.embedding = CodeMixEmbeddingAblation(no_lid_cfg, use_lang_id=False).to(device)

ckpt_path = f'{DRIVE_DIR}/checkpoints/ablation_no_lang_id_best.pt'
if os.path.exists(ckpt_path):
    no_lid_model.load_state_dict(torch.load(ckpt_path, map_location=device))
    no_lid_model.eval()

    no_lid_chatbot = CodeMixTChatbot(no_lid_model, f'{DRIVE_DIR}/tokenizer/codemix_bpe.model', no_lid_cfg)
    _, no_lid_preds, _ = generate_translations(df_test, no_lid_chatbot, 'No-LangID ablation')

    print('\n=== No-LangID Ablation Scores ===')
    no_lid_bleu, no_lid_chrf = score(no_lid_preds, ref_all, 'Full test set')

    no_lid_scores = {
        'Model': 'CodeMix-T (no LangID)',
        'BLEU (all)':      round(no_lid_bleu, 2),
        'chrF++ (all)':    round(no_lid_chrf, 2),
        'BLEU (Hinglish)': 'N/A',
        'BLEU (Tanglish)': 'N/A',
    }
    del no_lid_model
    torch.cuda.empty_cache()
else:
    print('No-LangID checkpoint not found. Run Phase 4 ablations first.')
    no_lid_scores = None

## Cell 8 — Final Results Table (Paper-Ready)

In [ ]:
rows = [codemix_scores, mbart_scores]
if no_lid_scores:
    rows.append(no_lid_scores)

results_df = pd.DataFrame(rows)
print('\n=== FINAL RESULTS TABLE ===')
print(results_df.to_string(index=False))

# Save as CSV and LaTeX for paper
results_df.to_csv(f'{DRIVE_DIR}/final_results.csv', index=False)

# Generate LaTeX table
latex = results_df.to_latex(index=False, caption='Translation results on the code-mixed test set.',
                             label='tab:results', escape=False)
with open(f'{DRIVE_DIR}/results_table.tex', 'w') as f:
    f.write(latex)

print(f'\nSaved: final_results.csv and results_table.tex')
print('\nLaTeX table:')
print(latex)

## Cell 9 — Error Analysis

In [ ]:
from sacrebleu.metrics import BLEU

# Compute sentence-level BLEU for each test sample
sentence_bleus = []
bleu_metric = BLEU(effective_order=True)

for src, pred, ref in zip(src_all, pred_all, ref_all):
    s = bleu_metric.sentence_score(pred, [ref])
    sentence_bleus.append(s.score)

df_errors = pd.DataFrame({
    'source':      src_all,
    'prediction':  pred_all,
    'reference':   ref_all,
    'bleu':        sentence_bleus
})

# Worst 20 translations
print('=== 20 WORST TRANSLATIONS ===')
worst = df_errors.nsmallest(20, 'bleu')
for _, row in worst.iterrows():
    print(f'  SRC:  {row["source"]}')
    print(f'  PRED: {row["prediction"]}')
    print(f'  REF:  {row["reference"]}')
    print(f'  BLEU: {row["bleu"]:.2f}\n')

# Best 10 translations
print('\n=== 10 BEST TRANSLATIONS ===')
best = df_errors.nlargest(10, 'bleu')
for _, row in best.iterrows():
    print(f'  SRC:  {row["source"]}')
    print(f'  PRED: {row["prediction"]}')
    print(f'  REF:  {row["reference"]}')
    print(f'  BLEU: {row["bleu"]:.2f}\n')

# BLEU distribution
print(f'\nSentence BLEU distribution:')
print(f'  Mean:   {df_errors["bleu"].mean():.2f}')
print(f'  Median: {df_errors["bleu"].median():.2f}')
print(f'  > 20:   {(df_errors["bleu"] > 20).sum()} samples')
print(f'  > 40:   {(df_errors["bleu"] > 40).sum()} samples')
print(f'  = 0:    {(df_errors["bleu"] == 0).sum()} samples (complete failures)')

df_errors.to_csv(f'{DRIVE_DIR}/error_analysis.csv', index=False)
print(f'\nSaved error analysis to error_analysis.csv')

## Cell 10 — Human Evaluation Template (100 samples)

In [ ]:
# Create a CSV for human evaluators to fill in
# Each sample gets rated on Adequacy (1-5) and Fluency (1-5)
# Standard practice for translation papers

import random
random.seed(42)

# Sample 100 sentences stratified by language
hi_sample = df_hinglish.sample(min(50, len(df_hinglish)), random_state=42)
ta_sample = df_tanglish.sample(min(50, len(df_tanglish)), random_state=42)
eval_sample = pd.concat([hi_sample, ta_sample]).sample(frac=1, random_state=42)

# Get our model's predictions for these
eval_preds = [chatbot.translate(row['source']) for _, row in tqdm(eval_sample.iterrows(), desc='Generating')]  

human_eval_df = pd.DataFrame({
    'ID':          range(1, len(eval_sample)+1),
    'Language':    eval_sample['language'].values,
    'Source':      eval_sample['source'].values,
    'Reference':   eval_sample['target'].values,
    'CodeMixT':    eval_preds,
    'Adequacy_1_5': '',    # Human fills in: Does meaning carry over? (1=No, 5=Perfect)
    'Fluency_1_5':  '',    # Human fills in: Is the English natural? (1=Unreadable, 5=Native)
    'Notes':        '',    # Human fills in: Any comments
})

path = f'{DRIVE_DIR}/human_evaluation_template.csv'
human_eval_df.to_csv(path, index=False)
print(f'Human evaluation template saved: {path}')
print(f'{len(human_eval_df)} samples ready for evaluation')
print('\nInstructions for evaluators:')
print('  Adequacy (1-5): 1=meaning lost, 3=partial, 5=fully preserved')
print('  Fluency  (1-5): 1=unreadable,  3=understandable, 5=native English')